# HOPE 교과서 PDF 구조화 실습

PDF 한 페이지를 이미지로 렌더링하고, 텍스트를 확보한 뒤 DocLayout-YOLO로 문서 요소를 탐지합니다. 이후 OCR·보정 규칙과 읽기 순서를 적용해 JSON과 시각화 이미지를 만듭니다.

**학습 흐름:** PDF → PyMuPDF 렌더링 → PDF 내장 텍스트 또는 PaddleOCR → DocLayout-YOLO → 보정 규칙 → 읽기 순서 → JSON

이 실습은 **구조화만 수행**하므로 OpenAI API 키가 필요하지 않습니다. 저작권과 개인정보 문제가 없는 PDF를 사용하세요.

## 1. 프로젝트와 라이브러리 설치

처음 한 번만 실행합니다. 설치와 모델 다운로드에는 몇 분이 걸릴 수 있습니다.

In [ ]:
import os
import subprocess

repo_dir = '/content/HOPE'
if not os.path.exists(repo_dir):
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/yyeonseoo/HOPE.git', repo_dir,
    ], check=True)

project_dir = os.path.join(repo_dir, 'project')
os.chdir(project_dir)
subprocess.run([
    'python', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt',
], check=True)
print('설치 완료:', os.getcwd())

## 2. 교과서 PDF 업로드

업로드한 파일은 현재 Colab 세션에만 저장되며 런타임이 종료되면 사라집니다.

In [ ]:
from pathlib import Path
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('PDF를 한 개 업로드해 주세요.')

pdf_name = next(iter(uploaded))
PDF_PATH = Path('/content') / pdf_name
PDF_PATH.write_bytes(uploaded[pdf_name])
print('업로드:', PDF_PATH)

## 3. 분석 조건 선택

- `PAGE_NUMBER`: 분석할 페이지 번호(1부터 시작)
- `DPI`: 150은 빠른 실습용, 200은 작은 글자가 많은 페이지에 권장
- `LAYOUT_MODE`: `enhanced`는 기본 보정, `raw`는 모델 원본, `unit3`는 좌표평면과 그래프 단원 실험용 보정입니다.

In [ ]:
PAGE_NUMBER = 1
DPI = 150
LAYOUT_MODE = 'enhanced'  # 'enhanced', 'raw', 'unit3' 중 선택

assert LAYOUT_MODE in {'enhanced', 'raw', 'unit3'}
MODEL_ID = 'hf:juliozhao/DocLayout-YOLO-DocStructBench'
OUTPUT_DIR = Path('/content/hope_output') / f'page_{PAGE_NUMBER:04d}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 4. 한 페이지 구조화 실행

텍스트 PDF는 내장 글자 좌표를 우선 사용합니다. 내장 텍스트가 부족한 스캔 PDF에서만 한국어 PaddleOCR 모델이 자동으로 실행됩니다. DocLayout-YOLO 가중치도 첫 실행 시 자동 다운로드됩니다.

In [ ]:
import sys
import time

sys.path.insert(0, '/content/HOPE/project/src')

from page_pipeline import process_single_page

started = time.perf_counter()
result = process_single_page(
    pdf_path=PDF_PATH,
    page_number=PAGE_NUMBER,
    work_dir=OUTPUT_DIR,
    dpi=DPI,
    yolo_model_path=MODEL_ID,
    lang='korean',
    prefer_pdf_text=True,
    model_only=(LAYOUT_MODE == 'raw'),
    correction_profile='unit3' if LAYOUT_MODE == 'unit3' else None,
)
elapsed = time.perf_counter() - started

print(f'분석 완료: {elapsed:.1f}초')
print('텍스트 출처:', result['ocr_source'])
print('탐지 블록 수:', len(result['page']['blocks']))

## 5. 원본과 구조화 시각화 비교

In [ ]:
from IPython.display import Image, display

print('원본 페이지')
display(Image(filename=str(result['page_image_path']), width=700))

print('구조화 결과: 번호는 권장 읽기 순서입니다.')
display(Image(filename=str(result['visualization_path']), width=700))

## 6. 블록 표와 JSON 확인

`bbox`는 렌더링된 페이지 이미지에서 `[x1, y1, x2, y2]`로 표현한 영역 좌표입니다. `score`는 DocLayout-YOLO 등의 탐지 신뢰도입니다.

In [ ]:
import json
import pandas as pd

blocks = result['page']['blocks']
rows = [{
    '읽기 순서': block.get('reading_order'),
    '블록 ID': block.get('block_id'),
    '유형': block.get('type'),
    '탐지 신뢰도': round(float(block.get('score', 0)), 3),
    'bbox': block.get('bbox'),
    '텍스트 미리보기': str(block.get('text', '')).replace('\n', ' ')[:80],
} for block in blocks]

display(pd.DataFrame(rows))

JSON_PATH = OUTPUT_DIR / 'layout.json'
payload = {
    'metadata': {
        'source_pdf': PDF_PATH.name,
        'page_number': PAGE_NUMBER,
        'dpi': DPI,
        'layout_mode': LAYOUT_MODE,
        'ocr_source': result['ocr_source'],
    },
    'page': result['page'],
}
JSON_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
print(JSON_PATH.read_text(encoding='utf-8')[:3000])

## 7. 결과 다운로드

In [ ]:
import shutil
from google.colab import files

archive_path = shutil.make_archive(
    f'/content/hope_page_{PAGE_NUMBER:04d}',
    'zip',
    root_dir=OUTPUT_DIR,
)
files.download(archive_path)

## 수업에서 확인할 질문

1. 원본 페이지의 실제 읽기 순서와 모델이 부여한 읽기 순서는 같은가요?
2. 표, 수식, Figure가 올바른 유형으로 분리되었나요?
3. 신뢰도가 낮은 블록은 실제로도 경계가 모호한가요?
4. `raw`와 `enhanced` 결과에서 추가·병합·재분류된 블록은 무엇인가요?
5. 자동 구조화 결과를 그대로 사용하지 않고 사람이 검수해야 하는 이유는 무엇인가요?